In [ ]:
import pandas as pd
import psycopg2
import os

conn = psycopg2.connect(
    host = 'localhost',
    port = '5432',
    database = 'postgres',
    user = 'postgres',
    password = ''
)

cur = conn.cursor()


query_create = """
-- 务必加分号
DROP TABLE IF EXISTS hot_drink_retail.final_simulation_base;

CREATE TABLE hot_drink_retail.final_simulation_base AS
SELECT
    s.name,
    s.location_type,
    s.lon,
    s.lat,
    t.*
FROM hot_drink_retail.beijing_drink_shops_processed AS s
CROSS JOIN  hot_drink_retail.temperature_trend_merged_data AS t;
"""


cur.execute(query_create)
conn.commit()


df_sql = pd.read_sql_query('SELECT * FROM hot_drink_retail.final_simulation_base', conn)

cur.close()
conn.close()

# 展示结果
display(df_sql.head())

In [ ]:
import pandas as pd
import psycopg2


conn = psycopg2.connect(database="postgres", user="postgres", host="localhost", port="5432")

df_shops = pd.read_sql("SELECT name, location_type FROM hot_drink_retail.beijing_drink_shops_processed", conn)
df_trends = pd.read_sql("SELECT * FROM hot_drink_retail.temperature_trend_merged_data", conn)



df_shops['key'] = 1
df_trends['key'] = 1


df_master_pd = pd.merge(df_shops, df_trends, on='key').drop('key', axis=1)

print(f"门店数: {len(df_shops)} | 天数: {len(df_trends)} | 总行数: {len(df_master_pd)}")

# 4. 查看结果
display(df_master_pd.head())

conn.close()

In [3]:
import numpy as np

def get_sales_prediction(row):
    """业务逻辑：将气压、店型和热度趋势转化为一个销售数字"""
    base_sales = 100
    avg_index = (row['milktea_index'] + row['coffee_index']) / 2.0
    trend_factor = avg_index / 50.0 

    temp = row['tmin']
    location_type = row['location_type']
    
    # 默认不影响
    weather_impact = 1.0
    
    # 【核心逻辑】：针对不同店型设置不同的耐寒系数
    if temp < 2:
        if location_type == '脆弱：极度依赖天气':
            weather_impact = max(0.2, 1.0 + (temp * 0.05)) # 景区最怕冷
        elif location_type == '抗寒：重度商务刚需':
            weather_impact = max(0.9, 1.0 + (temp * 0.005)) # 职场店最硬
        elif location_type == '中庸：混合社区':
            weather_impact = max(0.2, 1.0 + (temp * 0.02)) # 社区店适中
        else:
            weather_impact = 1.0
            
    # 计算最终值 + 随机震荡(噪音)
    prediction = (base_sales * trend_factor * weather_impact) + np.random.normal(0, 5)
    return round(prediction, 1)

In [ ]:
df_sql['pre_sale'] = df_sql.apply(get_sales_prediction,axis=1)
df_sql['pre_sale'] = df_sql['pre_sale'].astype(int)  
display(df_sql.sort_values(by='tmin').head(10))

path = r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\processed\result_pre_sale.csv"
df_sql.to_csv(path, index=False, encoding='utf-8-sig')